# 버그 찾기 3 (보통): 가장 좋은 층

위치 힌트: **없음.** 모든 셀이 따로 보면 맞게 돌아간다. 결론 중 코드가 보장하지 않는 것을 말하는 문장을 찾아라.

데이터: **합성** 층별 활성값이다. 24층 × (도시 150개 × 참/거짓 1개씩) = 300문장, d=32. `X[l]`이 l번째 층이다.

**하는 법**
1. 위에서부터 끝까지 실행하고, 맨 아래 **결론**을 읽는다.
2. 코드나 설명 중 "말과 실제 동작이 다른 곳"을 찾아 **문장으로** 적는다(맨 아래 `내 답` 셀).
3. 고치면 숫자가 어떻게 바뀔지 **한 줄 예측**을 먼저 적는다.
4. 고쳐서 다시 돌린다. 문법이 막히면 고칠 내용을 문장으로 쓰고 `???`로 남겨 두면 Claude가 채운다.

이 노트북은 오류 없이 끝까지 돈다. 정답은 Claude에게 따로 있다.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

d = np.load("acts/p3.npz")
X, y, group = d["X"], d["y"], d["group"]
print(X.shape)

In [ ]:
def group_split(groups, fractions=(0.6, 0.2, 0.2), seed=0):
    """같은 group의 행이 서로 다른 split에 들어가지 않게 나눈다 (probe_lab.group_split과 같다)."""
    unique = np.unique(groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique)
    a = int(round(fractions[0] * len(unique)))
    b = int(round(fractions[1] * len(unique)))
    parts = [unique[:a], unique[a : a + b], unique[a + b :]]
    return [np.flatnonzero(np.isin(groups, p)) for p in parts]


def fit_mm(X, y):
    """MM probe: 방향 θ = μ₁ − μ₀, 경계는 두 평균의 중점. 점수 = X @ θ + b."""
    mu1, mu0 = X[y == 1].mean(0), X[y == 0].mean(0)
    theta = mu1 - mu0
    return theta, float(-theta @ (mu1 + mu0) / 2)


tr, va, te = group_split(group)  # group 단위라 같은 도시가 train/test에 동시에 들어가지 않는다
print(len(tr), len(va), len(te))

In [ ]:
val_auc, test_auc = [], []
for l in range(X.shape[0]):
    theta, b = fit_mm(X[l][tr], y[tr])
    val_auc.append(roc_auc_score(y[va], X[l][va] @ theta + b))
    test_auc.append(roc_auc_score(y[te], X[l][te] @ theta + b))

best = int(np.argmax(test_auc))  # 가장 잘 되는 층
print("층별 test AUROC:", np.round(test_auc, 2))
print(f"가장 좋은 층 L{best}: test AUROC {test_auc[best]:.3f}")

In [ ]:
# 그 층에서 순열 귀무분포: train 라벨을 섞어 다시 학습, test AUROC
rng = np.random.default_rng(0)
null = []
for _ in range(200):
    theta, b = fit_mm(X[best][tr], rng.permutation(y[tr]))
    null.append(roc_auc_score(y[te], X[best][te] @ theta + b))
null = np.array(null)
p = (1 + np.sum(null >= test_auc[best])) / (1 + len(null))
print(f"L{best} null: {null.mean():.3f} ± {null.std():.3f}   p = {p:.3f}")

## 결론

1. L23에서 test AUROC 0.66으로 가장 높다.
2. 그 층의 순열 귀무분포(0.50 ± 0.07)에 비해 p ≈ 0.01이다. 우연이 아니다.
3. group 단위로 나눴으니 누수도 없다.
4. 따라서 참/거짓 정보는 L23에 있다. 다음 실험(개입)은 L23에서 한다.

## 내 답

- 말과 실제 동작이 다른 곳:
- 그래서 틀린 결론 문장(번호):
- 고치면 어떻게 바뀔지 예측:
- 고친 뒤 실제 결과: